# 🎯 Guía de IA Generativa - Prompt Engineering

En este notebook aprenderás:
- ✅ Qué es Prompt Engineering y por qué es crucial
- ✅ Técnicas fundamentales: zero-shot, few-shot, chain-of-thought
- ✅ Estructuras de prompt efectivas para diferentes tareas
- ✅ Cómo integrar prompts en aplicaciones reales
- ✅ Patrones avanzados y optimización

**⚠️ IMPORTANTE - SEGURIDAD:**
- ❌ NUNCA compartas tus API keys en este notebook
- ✅ Usa variables de entorno o widgets seguros

---
**📋 Requisitos previos:**
- Haber completado los Notebooks 1 y 2
- API key de Gemini (GRATIS) para los ejercicios interactivos
- Python 3.8+

# 0. SETUP Y CONTEXTO

## 📚 Instalar Librerías

In [ ]:
# 📦 Instalación de dependencias
import sys
import subprocess
print("🔧 Instalando paquetes necesarios...")
print("=" * 60)

packages = [
    ("google-genai", "🌟 Cliente de Google para Gemini (GRATIS)"),
    ("requests", "🌐 Para peticiones HTTP"),
    ("ipywidgets", "🎨 Widgets interactivos"),
    ("tiktoken", "🎫 Tokenizador para contar tokens"),
]

for package, descripcion in packages:
    try:
        print(f"📥 Instalando {package}...")
        print(f"   ℹ️  {descripcion}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✅ {package} instalado correctamente")
    except Exception as e:
        print(f"⚠️ Error instalando {package}: {e}")

print("=" * 60)
print("✨ ¡Instalación completada!")

## 📚 Importar Librerías

In [ ]:
# Importaciones estándar
import os
import sys
import json
import time
from datetime import datetime
from typing import List, Dict, Optional

# Visualización
from IPython.display import display, Markdown, HTML
import ipywidgets as widgets
from ipywidgets import Layout, Button, Box, VBox, HBox, Text, Textarea, Password, Dropdown

# Tokenización
try:
    import tiktoken
    tokenizer = tiktoken.get_encoding("cl100k_base")
    TIKTOKEN_AVAILABLE = True
except ImportError:
    TIKTOKEN_AVAILABLE = False

# Google Gemini
try:
    from google import genai
    from google.genai import types
    GEMINI_AVAILABLE = True
except ImportError:
    GEMINI_AVAILABLE = False
    print("⚠️ Google Gemini no disponible")

# Librerías para APIs
import requests

# Configuración SSL
VERIFICAR_SSL = False
if not VERIFICAR_SSL:
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

print("✅ Librerías importadas correctamente")
print(f"📊 Python: {sys.version.split()[0]}")
print(f"🕐 Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 🔐 Configuración de API Key

In [ ]:
# 🔐 Widget para API Key
print("🔑 Configuración de API Key")
print("=" * 60)
print("Este notebook NECESITA una API key de Gemini para los ejercicios.")
print("📎 Obtén tu key GRATIS en: https://makersuite.google.com/app/apikey")
print()

gemini_key_widget = Password(
    placeholder='AIza...',
    description='Gemini:',
    layout=Layout(width='500px')
)

save_button = Button(
    description='💾 Guardar y Conectar',
    button_style='success',
    layout=Layout(width='200px')
)

status_output = widgets.Output()

# Cliente global de Gemini
gemini_client = None

def save_and_connect(b):
    global gemini_client
    with status_output:
        status_output.clear_output()
        if gemini_key_widget.value and gemini_key_widget.value.startswith('AIza'):
            os.environ['GEMINI_API_KEY'] = gemini_key_widget.value
            try:
                gemini_client = genai.Client(api_key=gemini_key_widget.value)
                # Test rápido
                response = gemini_client.models.generate_content(
                    model='gemini-2.0-flash-lite',
                    contents='Di solo "OK"'
                )
                print("✅ ¡Conexión exitosa con Gemini!")
                print(f"   Respuesta de prueba: {response.text[:50]}")
            except Exception as e:
                print(f"❌ Error de conexión: {e}")
                gemini_client = None
        else:
            print("⚠️ Key inválida. Debe empezar con 'AIza'")

save_button.on_click(save_and_connect)

display(VBox([
    widgets.HTML("<h3>🔐 API Key de Gemini:</h3>"),
    gemini_key_widget,
    save_button,
    status_output
]))

## 🛠️ Funciones auxiliares

In [ ]:
# 🛠️ Función para llamar a Gemini de forma sencilla

def llamar_gemini(prompt: str, modelo: str = "gemini-2.0-flash-lite", max_tokens: int = 1000) -> str:
    """
    Llama a la API de Gemini y devuelve la respuesta.
    
    Args:
        prompt: El texto del prompt
        modelo: Modelo a usar (default: gemini-2.0-flash-lite, más rápido y con más cuota)
        max_tokens: Máximo de tokens en la respuesta
    
    Returns:
        Texto de la respuesta
    """
    if gemini_client is None:
        return "❌ ERROR: Primero configura tu API key en la celda anterior"
    
    try:
        response = gemini_client.models.generate_content(
            model=modelo,
            contents=prompt,
            config=types.GenerateContentConfig(
                max_output_tokens=max_tokens,
                temperature=0.7
            )
        )
        return response.text
    except Exception as e:
        return f"❌ ERROR: {str(e)}"

def comparar_prompts(prompt1: str, prompt2: str, titulo1: str = "Prompt A", titulo2: str = "Prompt B"):
    """
    Compara dos prompts ejecutándolos y mostrando los resultados lado a lado.
    """
    print("🔄 Ejecutando ambos prompts...")
    print()
    
    # Contar tokens
    tokens1 = len(tokenizer.encode(prompt1)) if TIKTOKEN_AVAILABLE else "N/A"
    tokens2 = len(tokenizer.encode(prompt2)) if TIKTOKEN_AVAILABLE else "N/A"
    
    # Ejecutar
    inicio1 = time.time()
    respuesta1 = llamar_gemini(prompt1)
    tiempo1 = time.time() - inicio1
    
    inicio2 = time.time()
    respuesta2 = llamar_gemini(prompt2)
    tiempo2 = time.time() - inicio2
    
    # Mostrar resultados
    print(f"{'='*60}")
    print(f"📝 {titulo1}")
    print(f"{'='*60}")
    print(f"🎫 Tokens: {tokens1} | ⏱️ Tiempo: {tiempo1:.2f}s")
    print(f"\n📋 PROMPT:")
    print(f"{prompt1[:200]}{'...' if len(prompt1) > 200 else ''}")
    print(f"\n📤 RESPUESTA:")
    print(respuesta1)
    print()
    print(f"{'='*60}")
    print(f"📝 {titulo2}")
    print(f"{'='*60}")
    print(f"🎫 Tokens: {tokens2} | ⏱️ Tiempo: {tiempo2:.2f}s")
    print(f"\n📋 PROMPT:")
    print(f"{prompt2[:200]}{'...' if len(prompt2) > 200 else ''}")
    print(f"\n📤 RESPUESTA:")
    print(respuesta2)

print("✅ Funciones auxiliares cargadas")

## 📝 Recordatorio: ¿Qué es un Prompt?

Un **prompt** es la instrucción que le damos a un LLM. Es nuestra forma de "programar" el comportamiento del modelo.

```
┌─────────────────────────────────────────┐
│            ANATOMÍA DE UN PROMPT        │
├─────────────────────────────────────────┤
│                                         │
│  👤 ROLE (opcional):                    │
│     "Eres un experto en..."             │
│                                         │
│  📋 CONTEXTO (opcional):                │
│     Información de fondo relevante      │
│                                         │
│  🎯 TAREA (obligatorio):                │
│     Lo que quieres que haga             │
│                                         │
│  📐 FORMATO (opcional):                 │
│     Cómo quieres la respuesta           │
│                                         │
│  📝 EJEMPLOS (opcional):                │
│     Demostraciones del output esperado  │
│                                         │
└─────────────────────────────────────────┘
```

---
# 🎓 1. TÉCNICAS FUNDAMENTALES

## 1.1 Zero-Shot Prompting

El **zero-shot** es la técnica más básica: pides algo al modelo **sin dar ejemplos**.

El modelo usa solo su conocimiento previo (del entrenamiento) para responder.

### ✅ Cuándo usar:
- Tareas simples y comunes
- Cuando el modelo entiende bien la tarea
- Para prototipar rápidamente

### ❌ Limitaciones:
- Puede no entender formatos específicos
- Menos control sobre el output
- Inconsistencia en tareas complejas

In [ ]:
# 🎯 Ejemplo Zero-Shot

prompt_zero_shot = """Clasifica el sentimiento del siguiente texto como POSITIVO, NEGATIVO o NEUTRO:

Texto: "El producto llegó tarde pero la calidad es excelente."

Sentimiento:"""

print("📝 ZERO-SHOT PROMPTING")
print("=" * 50)
print("Sin ejemplos - el modelo infiere qué hacer")
print()
print("📋 PROMPT:")
print(prompt_zero_shot)
print()
print("📤 RESPUESTA:")
respuesta = llamar_gemini(prompt_zero_shot)
print(respuesta)

## 1.2 Few-Shot Prompting

El **few-shot** proporciona **ejemplos** al modelo antes de la tarea real.

Esto "enseña" al modelo el formato y estilo esperado.

### ✅ Cuándo usar:
- Formatos de output específicos
- Tareas con patrones claros
- Clasificaciones personalizadas
- Cuando zero-shot falla

### 📊 Variantes:
- **One-shot**: 1 ejemplo
- **Few-shot**: 2-5 ejemplos
- **Many-shot**: 5+ ejemplos (raro, consume muchos tokens)

In [ ]:
# 🎯 Ejemplo Few-Shot

prompt_few_shot = """Clasifica el sentimiento del texto como POSITIVO, NEGATIVO o NEUTRO.

Texto: "Me encanta este restaurante, la comida es deliciosa"
Sentimiento: POSITIVO

Texto: "El servicio fue terrible, nunca volveré"
Sentimiento: NEGATIVO

Texto: "El local está en la calle Mayor número 5"
Sentimiento: NEUTRO

Texto: "El producto llegó tarde pero la calidad es excelente."
Sentimiento:"""

print("📝 FEW-SHOT PROMPTING")
print("=" * 50)
print("3 ejemplos que guían al modelo")
print()

# Comparar con zero-shot
comparar_prompts(
    prompt_zero_shot, 
    prompt_few_shot,
    "Zero-Shot (sin ejemplos)",
    "Few-Shot (3 ejemplos)"
)

## 1.3 Chain-of-Thought (CoT)

**Chain-of-Thought** hace que el modelo "piense paso a paso" antes de responder.

Esto mejora dramáticamente el rendimiento en tareas de **razonamiento**.

### 🧠 ¿Por qué funciona?
- Descompone problemas complejos
- Reduce errores de "salto" lógico
- El modelo puede "verificar" su razonamiento

### ✅ Cuándo usar:
- Matemáticas y lógica
- Problemas multi-paso
- Análisis complejo
- Toma de decisiones

In [ ]:
# 🎯 Ejemplo Chain-of-Thought

problema = "María tiene 3 manzanas. Le da la mitad a Juan. Luego compra 4 más. ¿Cuántas manzanas tiene María ahora?"

prompt_directo = f"{problema}\n\nRespuesta:"

prompt_cot = f"""{problema}

Piensa paso a paso:
1. Primero, ¿cuántas manzanas tiene María inicialmente?
2. ¿Cuántas le da a Juan?
3. ¿Cuántas le quedan después de dar?
4. ¿Cuántas compra?
5. ¿Cuál es el total final?

Respuesta:"""

print("🧠 CHAIN-OF-THOUGHT PROMPTING")
print("=" * 50)
print("Forzar al modelo a razonar paso a paso")
print()

comparar_prompts(
    prompt_directo,
    prompt_cot,
    "Respuesta Directa",
    "Chain-of-Thought"
)

In [ ]:
# 🎯 CoT Simplificado: "Piensa paso a paso"

# A veces basta con añadir esta frase mágica

problema_logica = """En una carrera, adelantas al que va segundo. ¿En qué posición quedas?"""

prompt_sin_cot = problema_logica + "\n\nRespuesta:"

prompt_con_cot = problema_logica + "\n\nPiensa paso a paso antes de responder.\n\nRespuesta:"

print("🧠 COT SIMPLIFICADO: 'Piensa paso a paso'")
print("=" * 50)
print()

comparar_prompts(
    prompt_sin_cot,
    prompt_con_cot,
    "Sin indicación de razonar",
    "Con 'Piensa paso a paso'"
)

## 1.4 Role Prompting

Asignar un **rol** o **persona** al modelo cambia su comportamiento y tono.

### 🎭 Ejemplos de roles útiles:
- "Eres un profesor experto en..."
- "Actúa como un revisor de código senior"
- "Eres un asistente médico (no das diagnósticos)"
- "Eres un copywriter creativo"

In [ ]:
# 🎭 Ejemplo Role Prompting

pregunta = "Explica qué es una API REST"

prompt_sin_rol = pregunta

prompt_con_rol = """Eres un profesor de programación experto en explicar conceptos técnicos 
a estudiantes principiantes. Usas analogías simples y ejemplos cotidianos. 
Evitas jerga técnica innecesaria.

Estudiante: Explica qué es una API REST

Profesor:"""

print("🎭 ROLE PROMPTING")
print("=" * 50)
print()

comparar_prompts(
    prompt_sin_rol,
    prompt_con_rol,
    "Sin rol asignado",
    "Con rol de profesor"
)

---
# 🏗️ 2. ESTRUCTURAS DE PROMPT EFECTIVAS

## 2.1 Plantilla CRAFT

Una estructura mnemotécnica para crear prompts completos:

```
C - Context (Contexto)    → Información de fondo
R - Role (Rol)            → Quién es el modelo
A - Action (Acción)       → Qué debe hacer
F - Format (Formato)      → Cómo debe responder
T - Target (Objetivo)     → Para quién/qué propósito
```

In [ ]:
# 📝 Ejemplo Plantilla CRAFT

prompt_simple = "Escribe un email de seguimiento para un cliente"

prompt_craft = """[CONTEXTO]
El cliente (Juan García) compró un software de gestión hace 1 semana.
No ha respondido al email de bienvenida ni ha activado su cuenta.

[ROL]
Eres un especialista en customer success con tono amable y profesional.

[ACCIÓN]
Escribe un email de seguimiento para reactivar al cliente.

[FORMATO]
- Asunto llamativo
- Máximo 100 palabras
- Incluir un CTA (call-to-action) claro
- Tono cercano pero profesional

[OBJETIVO]
Que el cliente active su cuenta y empiece a usar el producto."""

print("📝 PLANTILLA CRAFT")
print("=" * 50)
print()

comparar_prompts(
    prompt_simple,
    prompt_craft,
    "Prompt vago",
    "Prompt CRAFT estructurado"
)

## 2.2 Delimitadores y Estructura Clara

Usar **delimitadores** ayuda al modelo a distinguir partes del prompt:

- Triple comillas: `"""texto"""`
- XML tags: `<contexto>texto</contexto>`
- Corchetes: `[SECCIÓN]`
- Separadores: `---` o `===`

In [ ]:
# 📐 Ejemplo: Importancia de delimitadores

texto_analizar = """El nuevo iPhone es caro pero tiene buena cámara. 
No me gusta el diseño. La batería dura bastante."""

prompt_sin_delimitador = f"""Extrae los aspectos positivos y negativos de esta reseña:
{texto_analizar}
Dame la lista."""

prompt_con_delimitador = f"""Analiza la siguiente reseña de producto.

<reseña>
{texto_analizar}
</reseña>

Extrae:
1. ASPECTOS POSITIVOS (lista con viñetas)
2. ASPECTOS NEGATIVOS (lista con viñetas)
3. PUNTUACIÓN GENERAL (1-5 estrellas)"""

print("📐 USO DE DELIMITADORES")
print("=" * 50)
print()

comparar_prompts(
    prompt_sin_delimitador,
    prompt_con_delimitador,
    "Sin estructura clara",
    "Con delimitadores XML"
)

## 2.3 Output Estructurado (JSON)

Para **integración con aplicaciones**, necesitas output en formato estructurado.

In [ ]:
# 📊 Ejemplo: Forzar output JSON

producto = "Zapatillas Nike Air Max - 120€ - Running"

prompt_texto = f"""Extrae la información de este producto:
{producto}"""

prompt_json = f"""Extrae la información de este producto y devuélvela en formato JSON.

Producto: {producto}

Responde SOLO con el JSON válido, sin explicaciones. Usa este esquema:
{{
    "nombre": "string",
    "marca": "string",
    "precio": number,
    "moneda": "string",
    "categoria": "string"
}}"""

print("📊 OUTPUT ESTRUCTURADO (JSON)")
print("=" * 50)
print()

comparar_prompts(
    prompt_texto,
    prompt_json,
    "Output libre",
    "Output JSON estructurado"
)

In [ ]:
# 🔧 Procesando JSON en código

prompt_extraccion = """Extrae las entidades del siguiente texto y devuelve JSON:

Texto: "Apple lanzará el iPhone 16 en septiembre de 2024 por 1199 dólares"

Responde SOLO con JSON válido:
{
    "empresa": "string",
    "producto": "string",
    "fecha": "string",
    "precio": number,
    "moneda": "string"
}"""

respuesta_json = llamar_gemini(prompt_extraccion)
print("📤 Respuesta del modelo:")
print(respuesta_json)
print()

# Intentar parsear el JSON
try:
    # Limpiar posibles caracteres extra
    json_limpio = respuesta_json.strip()
    if json_limpio.startswith("```json"):
        json_limpio = json_limpio[7:]
    if json_limpio.startswith("```"):
        json_limpio = json_limpio[3:]
    if json_limpio.endswith("```"):
        json_limpio = json_limpio[:-3]
    
    datos = json.loads(json_limpio.strip())
    print("✅ JSON parseado correctamente:")
    for key, value in datos.items():
        print(f"   {key}: {value}")
except json.JSONDecodeError as e:
    print(f"❌ Error parseando JSON: {e}")

---
# ⚙️ 3. TÉCNICAS AVANZADAS

## 3.1 Self-Consistency

Ejecutar el mismo prompt **múltiples veces** y elegir la respuesta más frecuente.

Útil para tareas donde puede haber variabilidad.

In [ ]:
# 🔄 Ejemplo Self-Consistency

from collections import Counter

prompt_clasificacion = """Clasifica el siguiente texto en UNA sola categoría:
- TECNOLOGIA
- DEPORTES  
- POLITICA
- ENTRETENIMIENTO

Texto: "El nuevo chip M4 de Apple promete revolucionar los portátiles gaming"

Responde SOLO con la categoría, una palabra:"""

print("🔄 SELF-CONSISTENCY")
print("=" * 50)
print("Ejecutando el mismo prompt 5 veces...")
print()

respuestas = []
for i in range(5):
    resp = llamar_gemini(prompt_clasificacion, max_tokens=20)
    respuestas.append(resp.strip().upper())
    print(f"   Intento {i+1}: {resp.strip()}")

# Contar frecuencias
conteo = Counter(respuestas)
mas_comun = conteo.most_common(1)[0]

print()
print(f"📊 Distribución: {dict(conteo)}")
print(f"✅ Respuesta final (más frecuente): {mas_comun[0]} ({mas_comun[1]}/5 veces)")

## 3.2 Prompt Chaining (Encadenamiento)

Dividir una tarea compleja en **pasos secuenciales**, usando la salida de uno como entrada del siguiente.

In [ ]:
# 🔗 Ejemplo Prompt Chaining

texto_original = """La empresa TechCorp ha anunciado resultados récord en el Q4 2025, 
con ingresos de 50 mil millones de dólares, un 25% más que el año anterior. 
Sin embargo, los costes operativos aumentaron un 15% debido a la expansión 
en mercados asiáticos. El CEO María González destacó la importancia de la 
inversión en IA como motor de crecimiento futuro."""

print("🔗 PROMPT CHAINING")
print("=" * 50)
print("Paso 1: Extraer datos clave")
print()

# PASO 1: Extraer datos
prompt_paso1 = f"""Extrae los datos numéricos y entidades del siguiente texto.
Formato: lista simple.

Texto: {texto_original}

Datos extraídos:"""

datos_extraidos = llamar_gemini(prompt_paso1)
print("📤 Datos extraídos:")
print(datos_extraidos)
print()

# PASO 2: Analizar sentimiento/tono
print("Paso 2: Analizar tono")
print()

prompt_paso2 = f"""Basándote en estos datos extraídos de una noticia:
{datos_extraidos}

¿El tono general es POSITIVO, NEGATIVO o MIXTO? Explica brevemente."""

analisis_tono = llamar_gemini(prompt_paso2)
print("📤 Análisis de tono:")
print(analisis_tono)
print()

# PASO 3: Generar resumen ejecutivo
print("Paso 3: Generar resumen ejecutivo")
print()

prompt_paso3 = f"""Genera un resumen ejecutivo de 2 frases para un inversor.

Datos clave: {datos_extraidos}
Tono detectado: {analisis_tono}

Resumen ejecutivo:"""

resumen = llamar_gemini(prompt_paso3)
print("📤 Resumen ejecutivo final:")
print(resumen)

## 3.3 Negative Prompting (Qué NO hacer)

A veces es útil especificar lo que el modelo **NO debe hacer**.

In [ ]:
# ❌ Ejemplo Negative Prompting

prompt_sin_restricciones = """Escribe una intro para un artículo sobre Python."""

prompt_con_restricciones = """Escribe una intro para un artículo sobre Python.

RESTRICCIONES (NO hagas esto):
- NO uses clichés como "En el mundo actual..."
- NO menciones que Python es "fácil de aprender"
- NO hagas listas de características
- NO uses más de 50 palabras

Sé original y directo."""

print("❌ NEGATIVE PROMPTING")
print("=" * 50)
print()

comparar_prompts(
    prompt_sin_restricciones,
    prompt_con_restricciones,
    "Sin restricciones",
    "Con restricciones negativas"
)

## 3.4 ReAct (Reasoning + Acting)

Combina **razonamiento** con **acciones**. El modelo explica su pensamiento Y decide qué hacer.

Base de muchos sistemas de agentes IA.

In [ ]:
# 🤖 Ejemplo patrón ReAct

prompt_react = """Eres un asistente que resuelve problemas paso a paso.
Para cada paso, sigue el formato:
PENSAMIENTO: [tu razonamiento]
ACCIÓN: [qué harías / qué herramienta usarías]
OBSERVACIÓN: [resultado esperado]

Problema: Un usuario quiere saber el clima en Madrid para planificar un viaje el próximo fin de semana.

Resuelve paso a paso:"""

print("🤖 PATRÓN ReAct")
print("=" * 50)
print("Razonamiento explícito + Acciones")
print()
print("📤 RESPUESTA:")
respuesta = llamar_gemini(prompt_react)
print(respuesta)

---
# 🎮 4. EJERCICIOS PRÁCTICOS

## 4.1 Ejercicio: Mejora este prompt de clasificación

In [ ]:
# 🎯 EJERCICIO 1: Mejora este prompt

# Este prompt es vago y produce resultados inconsistentes
prompt_malo = """Dime si este email es spam o no:
¡FELICIDADES! Has ganado un iPhone. Haz clic aquí para reclamar tu premio."""

print("❌ PROMPT ACTUAL (malo):")
print(prompt_malo)
print()
print("📤 Respuesta:")
print(llamar_gemini(prompt_malo))
print()
print("="*50)
print("🎯 TU TAREA: Mejora el prompt para que:")
print("   1. Use few-shot (da ejemplos)")
print("   2. Defina criterios claros de spam")
print("   3. Pida un formato específico de respuesta")
print("   4. Incluya nivel de confianza")

In [ ]:
# ✅ SOLUCIÓN EJERCICIO 1

prompt_mejorado = """Clasifica el siguiente email como SPAM o NO_SPAM.

Criterios de SPAM:
- Promesas de premios/dinero no solicitados
- Urgencia artificial ("¡AHORA!", "¡ÚLTIMO DÍA!")
- Solicitud de datos personales o clics sospechosos
- Errores ortográficos intencionales para evitar filtros

Ejemplos:
Email: "Recordatorio: tu factura de luz vence el día 15"
Clasificación: NO_SPAM
Confianza: 95%
Razón: Email transaccional legítimo

Email: "Príncipe nigeriano quiere compartir $10M contigo"
Clasificación: SPAM
Confianza: 99%
Razón: Estafa clásica de herencia

Email: "¡FELICIDADES! Has ganado un iPhone. Haz clic aquí para reclamar tu premio."
Clasificación:
Confianza:
Razón:"""

print("✅ PROMPT MEJORADO:")
print(prompt_mejorado[:300] + "...")
print()
print("📤 Respuesta:")
print(llamar_gemini(prompt_mejorado))

## 4.2 Ejercicio: Crea un prompt para extracción de datos

In [ ]:
# 🎯 EJERCICIO 2: Extracción de datos de CVs

cv_texto = """Juan Pérez García
Email: juan.perez@email.com | Tel: +34 612 345 678
Madrid, España

EXPERIENCIA:
- Data Scientist en Google (2022-presente)
- ML Engineer en Amazon (2019-2022)

EDUCACIÓN:
- Máster en Data Science, Universidad Politécnica de Madrid (2019)
- Grado en Matemáticas, UCM (2017)

SKILLS: Python, TensorFlow, SQL, Spark
"""

print("🎯 TU TAREA:")
print("Crea un prompt que extraiga la información del CV en JSON estructurado")
print()
print("📄 CV DE ENTRADA:")
print(cv_texto)
print()
print("📊 JSON ESPERADO (esquema):")
print("""{
    "nombre": "string",
    "contacto": {"email": "...", "telefono": "...", "ubicacion": "..."},
    "experiencia": [{"puesto": "...", "empresa": "...", "periodo": "..."}],
    "educacion": [{"titulo": "...", "institucion": "...", "año": ...}],
    "skills": ["..."],
    "años_experiencia_total": number
}""")

In [ ]:
# ✅ SOLUCIÓN EJERCICIO 2

prompt_cv = f"""Extrae la información del siguiente CV y devuélvela en formato JSON.

<cv>
{cv_texto}
</cv>

Instrucciones:
1. Calcula los años de experiencia total sumando todos los períodos
2. Normaliza los skills a minúsculas
3. Usa el año actual (2026) para calcular "presente"

Responde SOLO con JSON válido usando este esquema exacto:
{{
    "nombre": "string",
    "contacto": {{
        "email": "string",
        "telefono": "string",
        "ubicacion": "string"
    }},
    "experiencia": [
        {{
            "puesto": "string",
            "empresa": "string",
            "año_inicio": number,
            "año_fin": number | null
        }}
    ],
    "educacion": [
        {{
            "titulo": "string",
            "institucion": "string",
            "año": number
        }}
    ],
    "skills": ["string"],
    "años_experiencia_total": number
}}"""

print("✅ PROMPT DE EXTRACCIÓN:")
print()
print("📤 Respuesta:")
respuesta = llamar_gemini(prompt_cv)
print(respuesta)

## 4.3 Ejercicio: Prompt para aplicación real

In [ ]:
# 🎯 EJERCICIO 3: Sistema de respuestas automáticas para soporte

# Simular tickets de soporte
tickets = [
    "No puedo acceder a mi cuenta, dice que la contraseña es incorrecta",
    "¿Cuánto cuesta el plan premium?",
    "La aplicación se cierra sola cuando abro la sección de reportes",
    "Quiero cancelar mi suscripción"
]

# Prompt para sistema de soporte
prompt_soporte = """Eres un agente de soporte técnico de SoftwareXYZ.

Tu tarea es analizar el ticket del cliente y:
1. Clasificar la CATEGORÍA: ACCESO, FACTURACIÓN, BUG, CANCELACIÓN, OTRO
2. Determinar PRIORIDAD: ALTA, MEDIA, BAJA
3. Generar una RESPUESTA inicial empática y útil (máx 2 frases)
4. Sugerir ACCIÓN para el equipo interno

CONTEXTO DE LA EMPRESA:
- Plan básico: 9€/mes
- Plan premium: 29€/mes
- Para resetear contraseña: enlace en página de login
- Bugs se escalan al equipo de desarrollo
- Cancelaciones requieren confirmación por email

Responde en formato:
CATEGORÍA: [categoría]
PRIORIDAD: [prioridad]
RESPUESTA AL CLIENTE: [respuesta empática]
ACCIÓN INTERNA: [qué hacer]

TICKET: "{ticket}"
"""

print("🎫 SISTEMA DE CLASIFICACIÓN DE TICKETS")
print("=" * 60)

for i, ticket in enumerate(tickets, 1):
    print(f"\n📨 TICKET #{i}: {ticket}")
    print("-" * 40)
    respuesta = llamar_gemini(prompt_soporte.format(ticket=ticket))
    print(respuesta)

---
# 💼 5. PROMPTS PARA INTEGRACIÓN CON APLICACIONES

## 5.1 Plantilla para APIs de producción

In [ ]:
# 🏭 Plantilla de prompt para producción

class PromptTemplate:
    """Clase para gestionar prompts de forma estructurada."""
    
    def __init__(self, template: str, required_vars: List[str]):
        self.template = template
        self.required_vars = required_vars
    
    def format(self, **kwargs) -> str:
        """Formatea el template con las variables proporcionadas."""
        # Verificar variables requeridas
        missing = [v for v in self.required_vars if v not in kwargs]
        if missing:
            raise ValueError(f"Variables faltantes: {missing}")
        
        return self.template.format(**kwargs)
    
    def count_tokens(self, **kwargs) -> int:
        """Cuenta tokens del prompt formateado."""
        formatted = self.format(**kwargs)
        return len(tokenizer.encode(formatted))

# Ejemplo: Plantilla para resumen de documentos
resumen_template = PromptTemplate(
    template="""Resume el siguiente documento en {num_frases} frases.
Idioma de salida: {idioma}
Enfoque: {enfoque}

<documento>
{documento}
</documento>

Resumen:""",
    required_vars=["documento", "num_frases", "idioma", "enfoque"]
)

# Uso
documento_ejemplo = """La inteligencia artificial ha experimentado avances significativos 
en los últimos años. Los modelos de lenguaje como GPT y Gemini pueden generar texto 
coherente y realizar tareas complejas. Sin embargo, también plantean desafíos éticos 
sobre privacidad, sesgo y desinformación."""

prompt_final = resumen_template.format(
    documento=documento_ejemplo,
    num_frases=2,
    idioma="español",
    enfoque="técnico"
)

print("🏭 PLANTILLA DE PRODUCCIÓN")
print("=" * 50)
print(f"📊 Tokens del prompt: {resumen_template.count_tokens(documento=documento_ejemplo, num_frases=2, idioma='español', enfoque='técnico')}")
print()
print("📤 Respuesta:")
print(llamar_gemini(prompt_final))

## 5.2 Validación de outputs

In [ ]:
# ✅ Validación de respuestas JSON

def llamar_con_validacion(prompt: str, esquema_esperado: dict, reintentos: int = 3) -> dict:
    """
    Llama al modelo y valida que la respuesta sea JSON válido.
    Reintenta si falla.
    """
    for intento in range(reintentos):
        respuesta = llamar_gemini(prompt)
        
        # Limpiar respuesta
        json_str = respuesta.strip()
        if json_str.startswith("```"):
            json_str = json_str.split("```")[1]
            if json_str.startswith("json"):
                json_str = json_str[4:]
        if json_str.endswith("```"):
            json_str = json_str[:-3]
        
        try:
            datos = json.loads(json_str.strip())
            
            # Validar campos esperados
            campos_faltantes = [k for k in esquema_esperado if k not in datos]
            if campos_faltantes:
                print(f"⚠️ Intento {intento + 1}: Campos faltantes: {campos_faltantes}")
                continue
            
            print(f"✅ Respuesta válida en intento {intento + 1}")
            return datos
            
        except json.JSONDecodeError as e:
            print(f"⚠️ Intento {intento + 1}: JSON inválido - {e}")
    
    return None

# Ejemplo de uso
prompt_producto = """Extrae info del producto y devuelve JSON:
"MacBook Pro 14 pulgadas, chip M3, 16GB RAM, 512GB SSD, 2499€"

JSON con campos: nombre, procesador, ram_gb, almacenamiento_gb, precio_eur"""

esquema = {"nombre": str, "procesador": str, "ram_gb": int, "almacenamiento_gb": int, "precio_eur": int}

print("✅ VALIDACIÓN DE OUTPUTS JSON")
print("=" * 50)
resultado = llamar_con_validacion(prompt_producto, esquema)
if resultado:
    print("\n📊 Datos extraídos:")
    for k, v in resultado.items():
        print(f"   {k}: {v}")

---
# 📋 RESUMEN Y CHEATSHEET

## 🎯 Cheatsheet de Técnicas

| Técnica | Cuándo usar | Ejemplo clave |
|---------|-------------|---------------|
| **Zero-shot** | Tareas simples, prototipado | `"Traduce: Hola"` |
| **Few-shot** | Formato específico, clasificación | Dar 2-3 ejemplos antes |
| **Chain-of-Thought** | Razonamiento, matemáticas | `"Piensa paso a paso"` |
| **Role prompting** | Cambiar tono/perspectiva | `"Eres un experto en..."` |
| **Delimitadores** | Separar contexto de tarea | `<contexto>`, `[TAREA]` |
| **Output JSON** | Integración con apps | Pedir esquema específico |
| **Self-consistency** | Reducir variabilidad | Ejecutar N veces, votar |
| **Prompt chaining** | Tareas complejas | Dividir en pasos |
| **Negative prompting** | Evitar comportamientos | `"NO hagas X"` |

## 📐 Reglas de Oro

1. **Sé específico**: Cuanto más claro, mejor resultado
2. **Da contexto**: El modelo no sabe lo que tú sabes
3. **Define el formato**: Especifica cómo quieres la respuesta
4. **Itera**: Tu primer prompt raramente será el mejor
5. **Prueba con edge cases**: ¿Funciona con inputs raros?

---

## 📚 Próximo notebook

**Notebook 4: RAG (Retrieval Augmented Generation)**
- Chunking de documentos
- Embeddings para retrieval
- Búsqueda semántica
- Generación con contexto

---

✨ **¡Felicidades!** Has completado el Notebook 3. Ahora sabes crear prompts efectivos para cualquier tarea.